# kappa of the calibration walkers against their own uniform reference

## Question and why it matters
The probe's `kappa` for a walker that is uniform on its territory at every instant is `kappa_unif(a, beta) = 2 * int_{1/2}^{1} x^(-a(1-beta)) dx` (1.110 at a = 2/3, beta = 1/2). Census ants sit at a median of 0.81x that reference (single run per rule, 5-95% 0.44-1.33). Is that location ant-specific, i.e. do other walkers with a known growth law get a different ratio when each is compared with its *own* (a, beta)? Links: `README.md`, `NOTEBOOK.md`, `experiments/archive/2026-10-kappa-uniform-baseline`.

## Assumptions and definitions
- ratio = kappa / kappa_unif(a, beta), with a (area vs t) and beta (perimeter vs area) fitted per run by log-log regression over the window set, kappa = mean over those windows of (boundary_steps / (t/2)) / (perimeter / area). `t` in the probe CSV is the window *end* (window (t/2, t]).
- Walker territory is assumed to follow single power laws over the window; a local-exponent variant agrees to <= 0.006 (README).
- Ant reference numbers (median 0.814, IQR 0.664-0.957, 5-95% 0.443-1.332) are copied from the baseline experiment's `census_spread.txt`; they are single-run and rule-dependent, the walkers are 8-seed means.

## Provenance
Prints the git commit, date, seed and a SHA-256 for every input file.

In [1]:
import hashlib, subprocess, datetime
from pathlib import Path
import numpy as np, polars as pl

SEED = 2026101901  # first walker seed; walkers use 2026101901..08, noisy rotors 2026102001..08. Nothing in this notebook is random.
np.random.seed(SEED)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "AGENTS.md").exists())
EXP = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "NOTEBOOK.md").exists())
R = EXP / "results"
INPUTS = [R / f for f in ["probe_mem.csv", "probe_rotor.csv", "probe_nrotor.csv", "specs_mem.txt", "specs_rotor.txt", "specs_nrotor.txt", "commit.txt", "commit_nrotor.txt", "probe_bin_nrotor.sha256"]]
missing = [p for p in INPUTS if not p.exists()]
assert not missing, f"missing inputs: {missing}"
sha = subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=ROOT, capture_output=True, text=True).stdout.strip()
print("git", sha, "| run", datetime.datetime.now().isoformat(timespec="minutes"), "| seed", SEED)
print("producing commits:", (R / "commit.txt").read_text().strip(), "(walkers),", (R / "commit_nrotor.txt").read_text().strip(), "(noisy rotor)")
for p in INPUTS:
    print(hashlib.sha256(p.read_bytes()).hexdigest()[:16], p.relative_to(ROOT))

git 8b290d8 | run 2026-10-19T21:41 | seed 2026101901
producing commits: 34fbf45 (walkers), f414649 (noisy rotor)
5675800c874a07ae experiments/2026-10-19-kappa-walker-reference/results/probe_mem.csv
45111ec686d82774 experiments/2026-10-19-kappa-walker-reference/results/probe_rotor.csv
9817622151025a14 experiments/2026-10-19-kappa-walker-reference/results/probe_nrotor.csv
fdf3a78df47998f8 experiments/2026-10-19-kappa-walker-reference/results/specs_mem.txt
fa2b94309b348e7f experiments/2026-10-19-kappa-walker-reference/results/specs_rotor.txt
ae11c4630dd4bb40 experiments/2026-10-19-kappa-walker-reference/results/specs_nrotor.txt
08b696e3cc5c1ad9 experiments/2026-10-19-kappa-walker-reference/results/commit.txt
7c8d7a147bd0ca6f experiments/2026-10-19-kappa-walker-reference/results/commit_nrotor.txt
065e1adff6b86b98 experiments/2026-10-19-kappa-walker-reference/results/probe_bin_nrotor.sha256


## Data and method
`scripts/run.sh` (rw, pw:0.25/0.5/0.75 at `probe 23 16384`; rotor at `probe 27 8192`; 8 seeds each) and `scripts/run_nrotor.sh` (`nrotor:EPS:SEED`, 9 EPS x 8 seeds, `probe 27 8192`), then `scripts/analyse.py`, `scripts/analyse_nrotor.py`. The cells below recompute the tables from the committed CSVs with the same estimators.

In [2]:
def kunif(a, b):
    p = a * (1 - b); return 2 * (1 - 0.5 ** (1 - p)) / (1 - p)

def run_stats(g, lo, hi):
    w = g.filter((pl.col("k") >= lo) & (pl.col("k") <= hi))
    if w.height < hi - lo + 1: return None   # escaped runs end early; only complete windows count
    t = w["t"].to_numpy().astype(float); A = w["area"].to_numpy().astype(float); P = w["perimeter"].to_numpy().astype(float)
    kap = (w["boundary_steps"].to_numpy() / (t / 2)) / (P / A)
    a = np.polyfit(np.log(t), np.log(A), 1)[0]; b = np.polyfit(np.log(A), np.log(P), 1)[0]
    return dict(a=a, beta=b, kappa=kap.mean(), ku=kunif(a, b), ratio=kap.mean() / kunif(a, b),
                g=np.polyfit(w["k"].to_numpy().astype(float), np.log(kap), 1)[0], ratio_last=kap[-1] / kunif(a, b))

def table(d, groups, lo, hi):
    out = []
    for name, pre in groups:
        runs = sorted(r for r in d["rule"].unique() if r.startswith(pre))
        rows = [s for r in runs if (s := run_stats(d.filter(pl.col("rule") == r).sort("k"), lo, hi))]
        f = lambda k: np.array([x[k] for x in rows])
        out.append(dict(walker=name, window=f"{lo}..{hi}", n=len(rows), a=f("a").mean(), beta=f("beta").mean(), kappa=f("kappa").mean(),
                        kunif=f("ku").mean(), ratio=f("ratio").mean(), ratio_sd=f("ratio").std(ddof=1), drift_g=f("g").mean(), ratio_lastwindow=f("ratio_last").mean()))
    return pl.DataFrame(out)

mem = pl.read_csv(R / "probe_mem.csv"); rot = pl.read_csv(R / "probe_rotor.csv"); nrot = pl.read_csv(R / "probe_nrotor.csv")
pl.Config.set_tbl_rows(40); pl.Config.set_tbl_cols(12); pl.Config.set_float_precision(3)
table_walkers = pl.concat([table(mem, [("rw", "rw:"), ("pw:0.25", "pw:0.25:"), ("pw:0.5", "pw:0.5:"), ("pw:0.75", "pw:0.75:")], 12, 22),
                           table(rot, [("rotor", "rotor:")], 12, 22), table(rot, [("rotor", "rotor:")], 20, 26)])
table_walkers

walker,window,n,a,beta,kappa,kunif,ratio,ratio_sd,drift_g,ratio_lastwindow
str,str,i64,f64,f64,f64,f64,f64,f64,f64,f64
"""rw""","""12..22""",8,0.920,0.917,1.307,1.024,1.277,0.042,0.005,1.355
"""pw:0.25""","""12..22""",8,0.903,0.898,1.271,1.029,1.235,0.026,0.008,1.311
"""pw:0.5""","""12..22""",8,0.920,0.909,1.350,1.026,1.316,0.043,0.011,1.304
"""pw:0.75""","""12..22""",8,0.911,0.907,1.287,1.026,1.255,0.056,0.003,1.282
"""rotor""","""12..22""",8,0.730,0.575,1.858,1.102,1.686,0.035,0.019,1.827
"""rotor""","""20..26""",8,0.685,0.530,1.989,1.106,1.799,0.033,0.002,1.800


Memoryless walkers have ratio 1.24-1.32 (their own reference is ~1.03 because beta ~ 0.9, not 1.16), the rotor 1.7-1.8, and the mean-field walker (baseline experiment, not rerun) 1.01. **Shows:** every non-uniform walker tested lies above the ant IQR (0.66-0.96), so the ant median 0.81 is not the value a generic walker gets. **Does not show:** why, or that the ant is below these walkers for a common reason; walker values are 8-seed means against a single-run ant distribution, and the memoryless walkers sit at the ant's 95th percentile.

In [3]:
table_nrotor = table(nrot, [(f"eps={e}", f"nrotor:{e}:") for e in ["0", "0.01", "0.02", "0.05", "0.1", "0.2", "0.4", "0.7", "1.0"]], 12, 22)  # eps strings must match specs_nrotor.txt ("1.0", not "1")
table_nrotor

walker,window,n,a,beta,kappa,kunif,ratio,ratio_sd,drift_g,ratio_lastwindow
str,str,i64,f64,f64,f64,f64,f64,f64,f64,f64
"""eps=0""","""12..22""",8,0.734,0.588,1.852,1.099,1.685,0.020,0.018,1.801
"""eps=0.01""","""12..22""",8,0.871,0.708,2.800,1.082,2.588,0.086,0.099,4.149
"""eps=0.02""","""12..22""",8,0.905,0.782,2.557,1.063,2.407,0.165,0.079,3.439
"""eps=0.05""","""12..22""",8,0.935,0.859,2.196,1.042,2.109,0.150,0.053,2.635
"""eps=0.1""","""12..22""",8,0.931,0.878,1.900,1.035,1.836,0.090,0.038,2.172
"""eps=0.2""","""12..22""",7,0.936,0.902,1.663,1.029,1.617,0.050,0.024,1.856
"""eps=0.4""","""12..22""",7,0.922,0.898,1.465,1.029,1.424,0.050,0.018,1.539
"""eps=0.7""","""12..22""",8,0.933,0.920,1.339,1.023,1.309,0.034,0.008,1.403
"""eps=1.0""","""12..22""",8,0.931,0.924,1.290,1.022,1.262,0.032,0.010,1.310


**Shows:** the noisy-rotor ratio is non-monotone: 1.69 (EPS 0), a peak of ~2.6 at EPS 0.01, then monotone down to 1.26 at EPS 1; a is already 0.87 at EPS 0.01. Runs at EPS <= 0.05 drift (`drift_g` is the slope of ln kappa per octave; the census stationarity cut is 0.05). The minimum ratio is 1.26, so noise never brings it near 0.81. **Does not show:** a mechanism for the low-EPS peak (a non-stationary transient); and these are windows k = 12..22, not the census window (noisy walkers leave the 8192 board at 2^23-2^26 steps, so k = 20..26 exists only for EPS 0 and 0.01).

In [4]:
# Census window: experiments/_lib/analyse_probe.py fits exponents over k = 20..26 but takes the kappa of the LAST
# window (k = 26), not the window mean used here. For the rotor, last-window ratio vs the 20..26 window mean:
table(rot, [("rotor", "rotor:")], 20, 26).select("walker", "window", "kappa", "ratio", "ratio_lastwindow")

walker,window,kappa,ratio,ratio_lastwindow
str,str,f64,f64,f64
"""rotor""","""20..26""",1.989,1.799,1.800


## Caveats and controls
- Registered predictions 1-3 (memoryless, rotor, location) held. For the noisy-rotor scan, (i) monotone decrease and (iii) a <= 0.75 up to EPS 0.1 **failed**; (ii) no ratio below 1, (iv) EPS 1 ~ pw:0.5 and (v) EPS 0 = rotor held.
- A first analysis took the window start for the window end and halved kappa (caught because it disagreed with an independent script, see `NOTEBOOK.md`). A second dropped every run with `escaped=true`, a run-level flag (README, deviation). Neither touched data.
- Window mismatch: memoryless and noisy walkers use k = 12..22 (a ~ 0.9 is not asymptotic); the rotor's ratio changes by 0.11 between k = 12..22 and 20..26. The census takes the last window k = 26, so walkers at k <= 22 are not window-matched to the ants.
- The mean-field anchor and the ant distribution come from `2026-10-kappa-uniform-baseline` and were not rerun here.

## What this changes, next steps, open questions
The ant deficit (median 0.81) is not reproduced by memoryless walkers, the rotor, or any rotor/noise mixture tested; it must come from something these stochastic walkers lack (a deterministic, rule-written colour field), or be an artefact of single-run noise. Not run: memoryless walkers at census-matched windows (README step 1), the occupancy-layer calibration (step 3), ant ratios averaged over seeds (step 4).